# 02 --- Pearson correlation

This notebook reproduces:

- **Table 8** of the manuscript: correlation matrix of recognition-module latencies.
- **Table 11** of the manuscript: Pearson correlation of Statement (5) with the other variables.

**Inputs:**
- `../data/raw/recognition-latencies.csv`
- `../data/raw/likert-effectiveness.csv`

**Outputs:**
- `../data/processed/correlation-matrix-latencies.csv`
- `../data/processed/correlation-statement5.csv`

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## Table 8 --- Correlation matrix of latencies

For each of the four AI modules the failed observations (`success = 0`) are replaced by the mean of the successful observations of the same module (as documented in the manuscript). This procedure keeps the total sample at $n = 16$ per module.

In [ ]:
df_lat = pd.read_csv('../data/raw/recognition-latencies.csv')
modules = ['person', 'facial', 'drowsiness', 'object']
for m in modules:
    ok = df_lat[df_lat[f'{m}_success'] == 1][f'{m}_latency_s']
    df_lat.loc[df_lat[f'{m}_success'] == 0, f'{m}_latency_s'] = ok.mean()

cols = {'person': 'People', 'facial': 'Facial Expressions',
        'drowsiness': 'Drowsiness', 'object': 'Objects'}
corr = df_lat[[f'{m}_latency_s' for m in modules]].corr(method='pearson').round(3)
corr.columns = [cols[m] for m in modules]
corr.index = [cols[m] for m in modules]
corr.to_csv('../data/processed/correlation-matrix-latencies.csv')
corr

## Table 11 --- Correlations of Statement (5) with the other variables

In [ ]:
df = pd.read_csv('../data/raw/likert-effectiveness.csv', comment='#')

target = 'statement_5'
candidates = ['tutor_age', 'supervision_time', 'student_age',
              'statement_1', 'statement_2', 'statement_3', 'statement_4',
              'statement_6', 'statement_7']

rows = []
for var in candidates:
    x = df[var].dropna()
    y = df.loc[x.index, target].dropna()
    common = x.index.intersection(y.index)
    if len(common) < 3:
        continue
    r, p = stats.pearsonr(x.loc[common], y.loc[common])
    abs_r = abs(r)
    if abs_r < 0.3:
        rel = 'weak'
    elif abs_r < 0.5:
        rel = 'moderate'
    elif abs_r < 0.7:
        rel = 'strong'
    else:
        rel = 'very strong'
    rows.append({'variable': var, 'pearson_r': round(r, 4),
                 'p_value': round(p, 4), 'strength': rel})

result = pd.DataFrame(rows)
result.to_csv('../data/processed/correlation-statement5.csv', index=False)
result